# 레이어 정규화 (Layer Normalization): 훈련 안정화
## 레이어 정규화의 수학과 효과

- **Tutorial ID**: `adv-1-1` | **Section ID**: `adv-1-1-1`

---

## 📚 이 노트북에서 배울 것들

| 파트 | 주제 | 핵심 질문 |
|------|------|----------|
| **0** | 사전 준비 | 어떤 도구가 필요한가? |
| **1** | 왜 정규화가 필요한가? | 없으면 어떤 문제가 생기나? |
| **2** | 통계 기초 | 평균·분산이란 무엇인가? |
| **3** | 레이어 정규화 수학 | 공식은 어떻게 계산하나? |
| **4** | 학습 파라미터 γ, β | 정규화를 "되돌릴" 수 있게 하는 이유는? |
| **5** | 분포 시각화 | 정규화 전후 분포가 어떻게 다른가? |
| **6** | RMSNorm | LayerNorm의 단순화 버전은 무엇인가? |
| **7** | Pre-LN vs Post-LN | 트랜스포머 어디에 정규화를 놓나? |
| **8** | 가중치 접기 | 추론 시 어떻게 최적화하나? |

> 💡 **읽는 법**:
> - 코드를 실행하기 전에 **`#` 주석을 먼저 읽으세요**
> - 숫자보다 **값의 변화 방향**에 집중하세요
> - 새 개념이 나오면 바로 위 마크다운 셀을 다시 확인하세요


In [ ]:
# ============================================================
# PART 0: 사전 준비 — 필요한 라이브러리 불러오기
# ============================================================
# numpy     : 행렬·벡터 계산 (이 노트북의 핵심 도구)
# matplotlib: 그래프 시각화
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# 한국어 폰트 설정 (환경마다 다를 수 있음)
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지
try:
    plt.rcParams["font.family"] = "NanumGothic"
except Exception:
    pass

# 재현 가능한 결과를 위해 랜덤 시드 고정
# ─ 시드(seed) = 랜덤 숫자 생성의 "시작점"
# ─ 같은 시드 → 실행할 때마다 같은 랜덤 숫자
np.random.seed(42)

print("✅ 모든 라이브러리를 성공적으로 불러왔습니다!")
print(f"   NumPy 버전: {np.__version__}")


---
## PART 1: 왜 정규화가 필요한가?

### 문제: 깊은 신경망에서 값이 불안정해진다

딥러닝 모델은 여러 층(layer)을 쌓아 만듭니다.

```
입력 → [Layer 1] → [Layer 2] → [Layer 3] → ... → [Layer N] → 출력
```

학습이 진행되면서 가중치(weight)가 업데이트될 때,
**각 층에 들어오는 입력의 분포가 계속 변합니다**.
이를 **내부 공변량 이동 (Internal Covariate Shift)** 이라 합니다.

### 왜 이게 문제인가?

| 증상 | 설명 |
|------|------|
| **그래디언트 소실** | 값이 너무 작으면 역전파 시 그래디언트가 0에 수렴 → 학습 불가 |
| **그래디언트 폭발** | 값이 너무 크면 그래디언트가 무한히 커짐 → 학습 실패 |
| **학습 불안정** | 층마다 다른 분포를 받아 매번 다시 적응해야 함 |

### 해결책: 정규화

각 층의 출력을 **평균≈0, 분산≈1** 로 조정해줍니다.

```
정규화 없이: [Layer 1] → (분포가 제멋대로 변함) → [Layer 2]
정규화 있이: [Layer 1] → 정규화(평균0, 분산1) → [Layer 2]
```

> 🎯 **핵심**: 정규화 = "매 층마다 값의 스케일을 리셋해주는 것"


In [ ]:
# ============================================================
# PART 1: 정규화 없이 깊은 네트워크를 통과하면 어떤 일이?
# ============================================================
# 실제 Attention/FFN 대신 간단한 행렬 곱(W @ x)으로 단순화합니다.
# 행렬 곱 = 하나의 층을 통과하는 것
# ============================================================

print("=" * 60)
print("❌ 문제 상황: 정규화 없이 10층 통과")
print("=" * 60)
print()

d = 64         # 벡터의 차원 (각 토큰의 특징 수)
n_layers = 10  # 시뮬레이션할 층 수

np.random.seed(42)

# 초기 입력: 표준 정규분포 (평균0, 표준편차1)
x = np.random.randn(d)
print(f"📌 초기 입력 (d={d}차원 벡터):")
print(f"   평균:     {np.mean(x):.4f}")
print(f"   표준편차: {np.std(x):.4f}")
print()

print(f"{'층':>4} | {'평균':>10} | {'표준편차':>10} | {'최대|값|':>10} | 상태")
print("-" * 60)

for i in range(n_layers):
    # 가중치 행렬: 표준편차 1/sqrt(d)로 초기화 (Xavier 초기화와 유사)
    W = np.random.randn(d, d) / np.sqrt(d)

    # 한 층 통과 (선형 변환)
    x = W @ x

    mean_val = np.mean(x)
    std_val  = np.std(x)
    max_val  = np.max(np.abs(x))

    status = "⚠️  불안정" if (std_val > 5 or std_val < 0.01) else "✅ 안정  "
    print(f"  {i+1:>2} | {mean_val:>10.4f} | {std_val:>10.4f} | {max_val:>10.4f} | {status}")

print()
print("→ 층이 깊어질수록 값의 스케일이 불규칙하게 변합니다!")
print("→ 이런 상황에서는 학습이 매우 불안정해집니다.")


In [ ]:
# ============================================================
# PART 1 (계속): 정규화를 추가하면 어떻게 달라지나?
# ============================================================
# simple_layer_norm은 "맛보기" 버전입니다.
# 자세한 구현은 PART 3에서 다룹니다.
# ============================================================

def simple_layer_norm(x, eps=1e-5):
    """
    간단한 레이어 정규화 (학습 파라미터 없는 버전)

    x  : 입력 벡터
    eps: 0으로 나누기 방지용 작은 값

    반환: 평균=0, 표준편차≈1로 정규화된 벡터
    """
    mean = np.mean(x)
    var  = np.var(x)
    return (x - mean) / np.sqrt(var + eps)


print("=" * 60)
print("✅ 해결책: 정규화와 함께 10층 통과")
print("=" * 60)
print()

# 같은 초기 입력 사용 (비교를 위해 시드 재설정)
np.random.seed(42)
x = np.random.randn(d)
print(f"📌 초기 입력 (d={d}차원):")
print(f"   평균: {np.mean(x):.4f}, 표준편차: {np.std(x):.4f}")
print()

print(f"{'층':>4} | {'정규화 후 평균':>18} | {'정규화 후 표준편차':>20} | 상태")
print("-" * 65)

for i in range(n_layers):
    W = np.random.randn(d, d) / np.sqrt(d)
    x = W @ x

    # ← 핵심 차이: 매 층 후 정규화 추가!
    x = simple_layer_norm(x)

    print(f"  {i+1:>2} | {np.mean(x):>18.8f} | {np.std(x):>20.8f} | ✅ 안정")

print()
print("→ 정규화 덕분에 평균≈0, 표준편차≈1을 유지합니다!")
print("→ 이제 학습이 훨씬 안정적으로 진행될 수 있습니다 🎉")


---
## PART 2: 통계 기초 — 평균, 분산, 표준편차

레이어 정규화 수식을 이해하려면 이 세 가지를 먼저 알아야 합니다.

### 2-1. 평균 (Mean) — μ

$$\mu = \frac{1}{d} \sum_{i=1}^{d} x_i$$

- 데이터의 **"중심" 위치**
- 예: `[1, 3, 5]` → 평균 = (1+3+5)/3 = **3**

### 2-2. 분산 (Variance) — σ²

$$\sigma^2 = \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2$$

- 데이터가 평균에서 얼마나 **퍼져 있는지** 측정
- 편차를 제곱하는 이유: 양수·음수가 서로 상쇄되지 않게
- 예: `[1, 3, 5]` → 분산 = (4+0+4)/3 ≈ **2.67**

### 2-3. 표준편차 (Standard Deviation) — σ

$$\sigma = \sqrt{\sigma^2}$$

- 분산의 제곱근 → 원래 데이터와 **같은 단위**로 해석 가능
- 예: 표준편차 = √2.67 ≈ **1.63**

### 2-4. 정규화 공식 (핵심!)

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

| 연산 | 효과 |
|------|------|
| `x_i - μ` | 중심을 0으로 이동 |
| `÷ √σ²` | 스케일을 1로 조정 |
| `+ ε` | 분모 0 방지 (ε ≈ 1e-5 = 0.00001) |

> 💡 **직관**: 정규화 = "중심을 0으로 옮기고, 단위를 표준화하는 것"


In [ ]:
# ============================================================
# PART 2: 평균·분산·정규화를 단계별로 직접 계산
# ============================================================

print("=" * 55)
print("📐 통계 기초: 단계별 손계산")
print("=" * 55)
print()

x_simple = np.array([1.0, 3.0, 5.0, 7.0, 9.0])
d_simple  = len(x_simple)

print(f"예시 벡터 x: {x_simple}  (원소 수 d={d_simple})")
print()

# ─── Step 1: 평균 ─────────────────────────────────────────
print("📌 Step 1: 평균(μ) 계산")
print("-" * 40)

total    = sum(x_simple)
mean_val = total / d_simple
print(f"  합계: {' + '.join(str(int(v)) for v in x_simple)} = {int(total)}")
print(f"  평균: {int(total)} ÷ {d_simple} = {mean_val}")
print(f"  np.mean() 확인: {np.mean(x_simple)}  ✅")
print()

# ─── Step 2: 분산 ─────────────────────────────────────────
print("📌 Step 2: 분산(σ²) 계산")
print("-" * 40)

deviations   = x_simple - mean_val    # 각 값 − 평균
squared_devs = deviations ** 2        # 편차 제곱
var_val      = np.mean(squared_devs)  # 평균

print(f"  편차 (x - μ):     {deviations}")
print(f"  편차 제곱 (x-μ)²: {squared_devs}")
print(f"  분산(평균):        {var_val:.4f}")
print(f"  np.var() 확인:    {np.var(x_simple):.4f}  ✅")
print()

# ─── Step 3: 표준편차 ─────────────────────────────────────
print("📌 Step 3: 표준편차(σ) 계산")
print("-" * 40)

std_val = np.sqrt(var_val)
print(f"  표준편차 = √{var_val:.4f} = {std_val:.4f}")
print(f"  np.std() 확인: {np.std(x_simple):.4f}  ✅")
print()

# ─── Step 4: 정규화 ───────────────────────────────────────
print("📌 Step 4: 정규화 (핵심!)")
print("-" * 40)

eps    = 1e-5
x_norm = (x_simple - mean_val) / np.sqrt(var_val + eps)

print(f"  공식: (x - μ) / √(σ² + ε)")
print(f"  정규화 전:  {x_simple}")
print(f"  정규화 후:  {np.round(x_norm, 4)}")
print()
print(f"  ✅ 검증:")
print(f"     평균:     {np.mean(x_norm):.2e}  ← 0에 매우 가깝죠?")
print(f"     표준편차: {np.std(x_norm):.6f}  ← 1에 매우 가깝죠?")
print()
print("💡 어떤 숫자로 시작해도, 정규화 후에는 항상 평균≈0, 표준편차≈1!")


---
## PART 3: 레이어 정규화 수학

### 3-1. Batch Norm vs Layer Norm — 정규화 방향이 다르다!

```
데이터: 3개의 샘플, 각 4개의 특징

         특징1  특징2  특징3  특징4
샘플1  [  2.1,  4.3,  1.2,  6.7 ]  ← Layer Norm: 이 행 방향으로 정규화
샘플2  [  3.5,  2.1,  8.3,  1.4 ]  ← Layer Norm: 이 행 방향으로 정규화
샘플3  [  1.1,  7.2,  4.5,  3.3 ]  ← Layer Norm: 이 행 방향으로 정규화
         ↑________________________↑
    Batch Norm: 이 열(배치) 방향으로 정규화
```

| | Batch Norm | **Layer Norm** |
|--|------------|----------------|
| 정규화 방향 | 배치(샘플들) 기준 | **한 샘플의 특징** 기준 |
| 사용 상황 | CNN, 이미지 | **트랜스포머, NLP** |
| 미니배치 필요? | ✅ 필요 | ❌ 샘플 1개로도 가능 |
| 가변 길이 시퀀스 | ⚠️ 어려움 | ✅ 자유로움 |

### 3-2. 레이어 정규화 전체 공식

$$\text{LayerNorm}(x) = \underbrace{\gamma}_{\text{스케일 (학습)}} \cdot \underbrace{\frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}}_{\text{정규화}} + \underbrace{\beta}_{\text{이동 (학습)}}$$

| 기호 | 이름 | 설명 | 초기값 |
|------|------|------|--------|
| **μ** | 평균 | 특징 차원의 평균 | — |
| **σ²** | 분산 | 특징 차원의 분산 | — |
| **ε** | 엡실론 | 0 나누기 방지 (1e-5) | — |
| **γ** | 감마 | 학습 가능한 스케일 | **1** |
| **β** | 베타 | 학습 가능한 이동 | **0** |

### 3-3. γ와 β가 왜 필요한가?

정규화 후엔 항상 평균=0, 분산=1이 됩니다.
하지만 모델에게 이 분포가 **항상 최적이 아닐 수 있습니다**.

γ와 β를 학습하면 **"필요하면 정규화를 부분적으로 되돌릴 수 있는"** 유연성을 갖습니다.

> 초기에는 `γ=1, β=0` → 정규화 그대로. 학습하면서 최적값으로 수렴.


In [ ]:
# ============================================================
# PART 3: 레이어 정규화 완전 구현 (γ, β 포함)
# ──────────────────────────────────────────────────────────
# ⚠️  이 함수는 이후 셀들에서도 계속 사용됩니다!
#     반드시 이 셀을 먼저 실행하세요.
# ============================================================

def layer_norm(x, gamma=None, beta=None, eps=1e-5):
    """
    레이어 정규화 (Layer Normalization)

    수식: LayerNorm(x) = γ × (x - μ) / √(σ² + ε) + β

    Parameters
    ----------
    x     : 입력 배열, shape = (..., d_model)
             ─ 마지막 차원(axis=-1)이 특징(feature) 차원
    gamma : 학습 가능한 스케일 파라미터 γ, shape = (d_model,)
             ─ None이면 γ=1 효과 (변화 없음)
    beta  : 학습 가능한 이동 파라미터 β, shape = (d_model,)
             ─ None이면 β=0 효과 (이동 없음)
    eps   : 0 나누기 방지 (기본값 1e-5)

    Returns
    -------
    정규화된 배열 (입력과 같은 shape)
    """
    # ─── Step 1: 마지막 차원(특징 차원)을 따라 평균 계산 ───
    # keepdims=True: 차원 유지 → 이후 브로드캐스팅을 위해 필요
    # 예) x.shape=(3,8) → mean.shape=(3,1) (토큰별 평균 보존)
    mean = np.mean(x, axis=-1, keepdims=True)

    # ─── Step 2: 분산 계산 ──────────────────────────────
    var = np.var(x, axis=-1, keepdims=True)

    # ─── Step 3: 정규화 ─────────────────────────────────
    # eps 이유: var=0(모든 값이 같은 경우) → 0으로 나누기 방지
    x_norm = (x - mean) / np.sqrt(var + eps)

    # ─── Step 4: 학습 파라미터 γ, β 적용 ────────────────
    if gamma is not None:
        x_norm = gamma * x_norm   # 원소별 곱셈 (스케일)
    if beta is not None:
        x_norm = x_norm + beta    # 원소별 덧셈 (이동)

    return x_norm


# ─── 실험 1: 1차원 벡터 ─────────────────────────────────
print("=" * 55)
print("📌 실험 1: 단순 벡터 (γ, β 없이)")
print("=" * 55)

x_1d    = np.array([1.0, 3.0, 5.0, 7.0, 9.0])
x_1d_ln = layer_norm(x_1d)

print(f"입력:         {x_1d}")
print(f"  평균={np.mean(x_1d):.1f}, 표준편차={np.std(x_1d):.4f}")
print(f"LayerNorm 후: {np.round(x_1d_ln, 4)}")
print(f"  평균={np.mean(x_1d_ln):.2e}  ← 0에 매우 가깝죠?")
print(f"  표준편차={np.std(x_1d_ln):.8f}  ← 1에 매우 가깝죠?")
print()

# ─── 실험 2: 배치 입력 (여러 토큰) ─────────────────────
print("=" * 55)
print("📌 실험 2: 배치 입력 (3개 토큰, 각 8차원)")
print("    실제 트랜스포머의 토큰 임베딩과 유사한 구조")
print("=" * 55)

d_model  = 8
n_tokens = 3

# 큰 값으로 만들어 정규화 효과를 뚜렷하게 확인
x_batch    = np.random.randn(n_tokens, d_model) * 10 + 5
x_batch_ln = layer_norm(x_batch)

print(f"입력 shape: {x_batch.shape}  = {n_tokens}토큰 × {d_model}차원")
print(f"정규화 전:  토큰별 평균={np.round(np.mean(x_batch, axis=-1), 1)}")
print(f"            토큰별 표준편차={np.round(np.std(x_batch, axis=-1), 2)}")
print(f"정규화 후:  토큰별 평균={np.round(np.mean(x_batch_ln, axis=-1), 6)}")
print(f"            토큰별 표준편차={np.round(np.std(x_batch_ln, axis=-1), 6)}")
print()
print("✅ 각 토큰이 독립적으로 정규화됩니다!")
print("   → 토큰 간 영향 없음 = 가변 길이 시퀀스 자유롭게 지원")


In [ ]:
# ============================================================
# PART 4: 학습 파라미터 γ(감마)와 β(베타)의 역할
# ============================================================
# γ와 β는 학습을 통해 최적값을 찾아가는 파라미터입니다.
# "정규화 후 이 분포가 최적이 아니라면, γ와 β로 조정하라!"
# ============================================================

print("=" * 55)
print("🎛️  γ(스케일)와 β(이동) 파라미터 실험")
print("=" * 55)
print()

x_demo      = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
x_norm_demo = layer_norm(x_demo)  # γ, β 없이 먼저 정규화

print(f"원본 데이터:  {x_demo}")
print(f"  평균={np.mean(x_demo):.1f}, 표준편차={np.std(x_demo):.4f}")
print(f"정규화 후:    {np.round(x_norm_demo, 4)}")
print(f"  평균≈0, 표준편차≈1")
print()

# ─── γ: 스케일 조정 ──────────────────────────────────────
print("📌 γ(감마)의 역할: 표준편차(스케일) 조정")
print("   γ < 1 → 값들이 더 촘촘해짐")
print("   γ = 1 → 변화 없음  ← 초기값")
print("   γ > 1 → 값들이 더 넓게 퍼짐")
print()

for g in [0.5, 1.0, 2.0, 3.0]:
    r = g * x_norm_demo
    print(f"  γ={g:.1f}: {np.round(r, 3)}  (표준편차={np.std(r):.3f})")
print()

# ─── β: 이동 ─────────────────────────────────────────────
print("📌 β(베타)의 역할: 평균 이동(Shift)")
print("   β < 0 → 전체 값을 아래로 이동")
print("   β = 0 → 이동 없음  ← 초기값")
print("   β > 0 → 전체 값을 위로 이동")
print()

for b in [-1.5, 0.0, 1.0, 2.0]:
    r = x_norm_demo + b
    print(f"  β={b:+.1f}: {np.round(r, 3)}  (평균={np.mean(r):.2f})")
print()

# ─── γ, β 함께: 원래 분포 완전 복원 ──────────────────────
print("📌 γ, β 함께: 정규화를 완전히 되돌리기")
print("   (모델이 '이 정규화가 필요 없다'고 학습한 극단적 예)")
print()

g_restore = np.std(x_demo)   # 원래 표준편차
b_restore = np.mean(x_demo)  # 원래 평균

x_restored = g_restore * x_norm_demo + b_restore
print(f"  γ={g_restore:.4f} (원래 표준편차), β={b_restore:.1f} (원래 평균)")
print(f"  복원 결과:  {np.round(x_restored, 4)}")
print(f"  원본 데이터: {x_demo}")
print(f"  원본 복원 성공? {np.allclose(x_restored, x_demo)} ✅")
print()
print("💡 정리:")
print("   ① 학습 초기: γ=1, β=0  → 정규화 결과 그대로")
print("   ② 학습 중  : γ, β가 태스크에 맞게 조금씩 조정됨")
print("   ③ 정규화의 안정성 + 유연성을 동시에 얻을 수 있습니다!")


In [ ]:
# ============================================================
# PART 5: 시각화 — 정규화 전후 분포 변화
# ============================================================
# 히스토그램: 값의 범위별로 몇 개나 해당하는지 막대로 표시
# ============================================================

print("=" * 55)
print("📊 시각화: 레이어 정규화 전후 분포 비교")
print("=" * 55)
print()

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle("레이어 정규화 전후 분포 비교", fontsize=14, fontweight="bold")

np.random.seed(0)
test_cases = [
    ("분포1: 평균+10, 표준편차=5",
     np.random.randn(2000) * 5 + 10),
    ("분포2: 평균-5, 표준편차=2",
     np.random.randn(2000) * 2 - 5),
    ("분포3: 두 봉우리 혼합",
     np.concatenate([np.random.randn(1000) - 5,
                     np.random.randn(1000) + 5])),
]
colors_b = ["salmon",    "lightgreen", "plum"]
colors_a = ["firebrick", "seagreen",   "purple"]

for idx, (title, x_s) in enumerate(test_cases):
    x_n = (x_s - np.mean(x_s)) / np.std(x_s)

    ax0 = axes[0, idx]
    ax0.hist(x_s, bins=60, color=colors_b[idx], alpha=0.75, edgecolor="white")
    ax0.axvline(np.mean(x_s), color="black", lw=2, ls="--",
                label=f"평균={np.mean(x_s):.1f}")
    ax0.set_title(f"정규화 전\n{title}", fontsize=9)
    ax0.set_xlabel("값"); ax0.set_ylabel("빈도"); ax0.legend(fontsize=8)

    ax1 = axes[1, idx]
    ax1.hist(x_n, bins=60, color=colors_a[idx], alpha=0.75, edgecolor="white")
    ax1.axvline(0, color="black", lw=2, ls="--", label="평균≈0")
    ax1.set_title(f"정규화 후\n평균≈{np.mean(x_n):.4f}, 표준편차≈{np.std(x_n):.4f}",
                  fontsize=9)
    ax1.set_xlabel("값"); ax1.set_ylabel("빈도"); ax1.legend(fontsize=8)
    ax1.set_xlim(-4.5, 4.5)

    print(f"케이스 {idx+1}: {title}")
    print(f"  정규화 전 → 평균: {np.mean(x_s):+.2f}, 표준편차: {np.std(x_s):.2f}")
    print(f"  정규화 후 → 평균: {np.mean(x_n):.2e}, 표준편차: {np.std(x_n):.6f}")
    print()

axes[0, 0].set_ylabel("정규화 전\n빈도", fontsize=10, fontweight="bold")
axes[1, 0].set_ylabel("정규화 후\n빈도", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()
print("→ 어떤 분포로 시작해도 정규화 후엔 항상 비슷한 모양이 됩니다!")


---
## PART 6: RMSNorm (Root Mean Square Normalization)

### 무엇인가?

LLaMA, Gemma, Mistral 등 **최신 LLM에서 사용하는 단순화된 정규화**입니다.

### 수식 비교

**LayerNorm:**
$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

**RMSNorm:**
$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\text{RMS}(x)}, \qquad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

### 단계별 차이

| 단계 | LayerNorm | RMSNorm |
|------|-----------|---------|
| ① 평균(μ) 계산 | ✅ 있음 | ❌ **없음** |
| ② 평균 빼기 | ✅ 있음 | ❌ **없음** |
| ③ RMS 계산 | ❌ 없음 | ✅ 있음 |
| ④ 나누기 | √분산으로 나눔 | RMS로 나눔 |
| β 파라미터 | ✅ 있음 | ❌ 없음 (보통) |

### RMS란?

**RMS = Root Mean Square = 제곱 평균의 제곱근**

$$\text{RMS}([3, 4, 0, 0]) = \sqrt{\frac{3^2 + 4^2 + 0^2 + 0^2}{4}} = \sqrt{\frac{25}{4}} = 2.5$$

직관: 벡터의 "평균적인 크기"를 측정

### 왜 빠른가?

평균 빼기와 분산 계산을 생략 → 연산 단계 감소.  
Zhang & Sennrich(2019) 연구에서 **평균 제거 단계가 성능에 크게 기여하지 않음**을 발견.  
실제 학습된 LLM에서 활성값의 평균이 0에 가까운 경향이 있기 때문입니다.


In [ ]:
# ============================================================
# PART 6: RMSNorm 구현 및 LayerNorm과 비교
# ============================================================

def rms_norm(x, gamma=None, eps=1e-5):
    """
    RMSNorm (Root Mean Square Layer Normalization)

    수식: RMSNorm(x) = γ × x / RMS(x)
          RMS(x) = √(mean(x²) + ε)

    LayerNorm과의 핵심 차이:
      - 평균(μ)을 빼지 않음  ← 핵심!
      - β(이동) 파라미터 없음
      - 단순하고 빠름

    Parameters
    ----------
    x     : 입력 배열
    gamma : 스케일 파라미터 γ
    eps   : 수치 안정성을 위한 작은 값
    """
    # RMS 계산
    # 1) 각 원소를 제곱
    # 2) 평균 계산 (axis=-1 : 특징 차원 방향)
    # 3) 제곱근 → RMS
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)

    # x를 RMS로 나눔 (평균을 빼지 않는다는 것이 LayerNorm과 다름!)
    x_norm = x / rms

    # 스케일 파라미터 γ 적용
    if gamma is not None:
        x_norm = gamma * x_norm

    return x_norm


# ─── 비교 1: 임의 벡터 ───────────────────────────────────
print("=" * 55)
print("📌 비교 1: 임의 벡터에서 LayerNorm vs RMSNorm")
print("=" * 55)

x_test = np.array([3.0, 4.0, 0.0, -2.0, 1.0, 5.0])
print(f"입력: {x_test}")
print(f"  평균     : {np.mean(x_test):.4f}  ← LayerNorm은 이 값을 뺌")
print(f"  RMS      : {np.sqrt(np.mean(x_test**2)):.4f}  ← RMSNorm은 이 값으로 나눔")
print()

x_ln  = layer_norm(x_test)
x_rms = rms_norm(x_test)

print(f"LayerNorm 결과: {np.round(x_ln, 4)}")
print(f"  평균={np.mean(x_ln):.4f}, 표준편차={np.std(x_ln):.4f}")
print()
print(f"RMSNorm   결과: {np.round(x_rms, 4)}")
print(f"  평균={np.mean(x_rms):.4f}, 표준편차={np.std(x_rms):.4f}")
print()
print("💡 두 결과가 다른 이유:")
print("   LayerNorm은 '중심을 0으로 옮긴 뒤' 나눕니다")
print("   RMSNorm은 '중심 이동 없이' RMS로만 나눕니다")
print()

# ─── 비교 2: 평균이 0이면 거의 같아진다 ─────────────────
print("=" * 55)
print("📌 비교 2: 입력 평균이 0인 경우 (두 방법이 거의 같아짐)")
print("=" * 55)

x_zero_mean = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])  # 평균 = 0
print(f"입력: {x_zero_mean}  (평균={np.mean(x_zero_mean):.0f})")
print()

x_ln_zm  = layer_norm(x_zero_mean)
x_rms_zm = rms_norm(x_zero_mean)

print(f"LayerNorm 결과: {np.round(x_ln_zm, 4)}")
print(f"RMSNorm   결과: {np.round(x_rms_zm, 4)}")
print(f"두 결과가 거의 같은가? {np.allclose(x_ln_zm, x_rms_zm, atol=1e-5)}")
print()
print("💡 실제 LLM 활성값의 평균이 0에 가까운 경향이 있어서")
print("   RMSNorm이 LayerNorm과 비슷한 성능을 내면서 더 빠른 것입니다!")

# ─── RMS 손계산 확인 ─────────────────────────────────────
print()
print("=" * 55)
print("📌 RMS 손계산 검증")
print("=" * 55)

x_rms_demo = np.array([3.0, 4.0, 0.0, 0.0])
rms_manual = np.sqrt((3**2 + 4**2 + 0**2 + 0**2) / 4)

print(f"입력: {x_rms_demo}")
print(f"RMS 계산: √((3²+4²+0²+0²)/4) = √(25/4) = √6.25 = {rms_manual:.4f}")
print(f"np 확인: {np.sqrt(np.mean(x_rms_demo**2)):.4f}  ✅")


---
## PART 7: Pre-LN vs Post-LN 아키텍처

트랜스포머에서 레이어 정규화를 **어느 위치**에 넣느냐에 따라 두 가지 방식이 있습니다.

### 트랜스포머 블록 기본 구조

각 트랜스포머 블록은 두 개의 서브레이어로 구성됩니다:
1. **Multi-Head Attention (MHA)**
2. **Feed-Forward Network (FFN)**

각 서브레이어에는 **잔차 연결(Residual Connection)** 이 적용됩니다:
```
출력 = SubLayer(입력) + 입력
         ^^^^^^^^^^^^   ^^^^^
         서브레이어 경로  잔차 경로 (Skip Connection)
```
잔차 연결의 장점: 그래디언트가 잔차 경로를 통해 **직접** 이전 층으로 전달됩니다.

---

### Post-LN (원래 Transformer, Vaswani et al. 2017)

```
              ┌───── SubLayer ─────┐
입력 x ───────┤                    ├──➕──→ LayerNorm → x_next
              └────────────────────┘   ↑
                                       x (잔차)
```

**특징**: LN이 **잔차 연결 이후** 적용  
**문제**: 잔차 경로를 통해 온 그래디언트도 LN을 통과 → 초기 학습 불안정

---

### Pre-LN (GPT-2 이후 대부분의 현대 LLM)

```
              ┌── LayerNorm ── SubLayer ──┐
입력 x ───────┤                           ├──➕──→ x_next
              └───────────────────────────┘   ↑
                                              x (잔차, LN 없이 직접!)
```

**특징**: LN이 **서브레이어 이전** 적용  
**장점**: 잔차 경로(x → 직접 합산)는 LN을 거치지 않음 → 그래디언트 고속도로!

---

### 핵심: 그래디언트 흐름 차이

```
Post-LN 역전파:
   ∂L/∂x_early ← [LN의 역전파] ← [ADD] ← [SubLayer의 역전파]
                   ^^^^^^^^^^^^
                   LN이 그래디언트를 "필터링" → 깊은 모델에서 소실 가능

Pre-LN 역전파:
   ∂L/∂x_early ← [ADD] ← {[SubLayer의 역전파] + [x 직접 경로]}
                                                    ^^^^^^^^^^^^^
                                                    LN 없이 직접 흐름! ✅
```

> 현대 LLM(GPT-2, GPT-3, LLaMA, Gemma 등)은 대부분 **Pre-LN** 사용


In [ ]:
# ============================================================
# PART 7: Pre-LN vs Post-LN 시뮬레이션
# ============================================================
# 실제 Attention/FFN 대신 간단한 선형 변환(W @ x)으로 단순화하여
# 두 아키텍처에서 값이 어떻게 달라지는지 비교합니다.
# ============================================================

print("=" * 60)
print("🏗️  Pre-LN vs Post-LN 아키텍처 비교")
print("=" * 60)
print()

d        = 16   # 특징 차원
n_layers = 8    # 층 수

np.random.seed(42)
# 서브레이어 역할을 대신할 가중치 행렬들
weights = [np.random.randn(d, d) * 0.3 for _ in range(n_layers)]

x_init = np.random.randn(d)
print(f"공통 초기 입력: 평균={np.mean(x_init):.4f}, 표준편차={np.std(x_init):.4f}")
print(f"차원(d)={d}, 층(n_layers)={n_layers}")
print()

# ─── Post-LN ─────────────────────────────────────────────
print("─" * 50)
print("❶ Post-LN | 구조: x → SubLayer → ADD → LN → x_next")
print("─" * 50)

x_post = x_init.copy()

for i in range(n_layers):
    W = weights[i]

    h = W @ x_post           # 1. 서브레이어 통과 (선형 변환)
    x_res = x_post + h       # 2. 잔차 연결 (ADD)
    x_post = layer_norm(x_res)  # 3. LayerNorm (잔차 연결 후!)

    print(f"  층 {i+1}: 출력 평균={np.mean(x_post):.4f}, 표준편차={np.std(x_post):.4f}")

print()

# ─── Pre-LN ──────────────────────────────────────────────
print("─" * 50)
print("❷ Pre-LN  | 구조: x → LN → SubLayer → ADD → x_next")
print("─" * 50)

x_pre = x_init.copy()

for i in range(n_layers):
    W = weights[i]

    x_normed = layer_norm(x_pre)  # 1. LayerNorm (서브레이어 전!)
    h = W @ x_normed              # 2. 서브레이어 통과
    x_pre = x_pre + h             # 3. 잔차 연결 (원래 x_pre와 더함)
    #                                    ↑ 중요: x_normed가 아닌 x_pre!
    #                                      잔차 경로에는 LN이 없음

    print(f"  층 {i+1}: 출력 평균={np.mean(x_pre):.4f}, 표준편차={np.std(x_pre):.4f}")

print()

# ─── 비교 설명 ────────────────────────────────────────────
print("=" * 60)
print("📊 핵심 차이 설명")
print("=" * 60)
print()
print("Post-LN의 잔차 연결:")
print("   x_post = LayerNorm(x + SubLayer(x))")
print("   → 잔차 경로(+x)가 있지만, 두 경로 합 후 LN을 거침")
print("   → 그래디언트도 LN을 거쳐야만 이전 층으로 전달됨")
print("   → 깊은 모델 초기 학습에서 불안정할 수 있음")
print()
print("Pre-LN의 잔차 연결:")
print("   x_pre = x + SubLayer(LayerNorm(x))")
print("   → 잔차 경로(+x)는 LN을 전혀 거치지 않음!")
print("   → 그래디언트가 잔차 경로를 통해 직접 이전 층으로 흐름")
print("   → 대규모 모델에서 더 안정적인 학습!")
print()
print("✅ GPT-2, LLaMA, Gemma 등 현대 LLM은 Pre-LN을 사용합니다.")


---
## PART 8: 가중치 접기 (Weight Folding)

### 배경: 추론 최적화

학습이 끝난 후 **배포(deploy) 단계**에서 파라미터는 고정됩니다.
고정된 상태에서 **추론(inference) 시간을 줄이기** 위한 최적화 기법입니다.

### 핵심 아이디어

LayerNorm 이후에 선형 변환(Linear Layer)이 바로 오는 경우:

```
x → LayerNorm(γ, β) → Linear(W) → 출력
```

수식으로 풀면:

$$\text{출력} = W \cdot (\gamma \odot \hat{x} + \beta)
= \underbrace{(W \cdot \text{diag}(\gamma))}_{W_{\text{folded}}} \cdot \hat{x}
+ \underbrace{W \cdot \beta}_{b_{\text{folded}}}$$

- $\hat{x}$: γ, β 없이 정규화만 한 값
- $W_{\text{folded}}$: W의 각 열(column)에 γ를 미리 곱한 가중치
- $b_{\text{folded}}$: β 효과를 미리 계산한 바이어스

### 직관적 그림

```
원래 방식 (매 추론마다):
  x̂  →  [γ 곱셈]  →  [β 덧셈]  →  [W 행렬곱]  →  출력
         ^^^^^^^^      ^^^^^^^^      ^^^^^^^^^^^
         불필요한 별도 연산들

가중치 접기 후 (매 추론마다):
  x̂  →  [W_folded 행렬곱 + b_folded]  →  출력
                                      
  (* W_folded = W × diag(γ) 는 배포 전 딱 한 번만 계산)
```

### 언제 쓸 수 있나?

| 상황 | 사용 가능? |
|------|-----------|
| 학습 중 | ❌ γ가 업데이트되므로 불가 |
| 추론 전용 배포 | ✅ 파라미터 고정 후 가능 |

> 실제로 양자화(Quantization), ONNX 내보내기 등의 도구들이 이 기법을 내부적으로 사용합니다.


In [ ]:
# ============================================================
# PART 8: 가중치 접기 — 구현 및 수학적 검증
# ============================================================

print("=" * 60)
print("⚡ 가중치 접기 (Weight Folding) 구현")
print("=" * 60)
print()

np.random.seed(7)
d = 5  # 간단한 예시를 위해 5차원 사용

# 임의 입력
x = np.array([2.0, -1.0, 4.0, 0.5, -3.0])

# 학습 완료 후 고정된 LayerNorm 파라미터
gamma = np.array([1.5,  0.8, 2.0, 1.2,  0.6])
beta  = np.array([0.3, -0.1, 0.0, 0.5, -0.2])

# LayerNorm 바로 뒤의 선형 층 가중치 (고정)
W = np.array([
    [ 1.0,  0.5, -0.5,  0.2,  1.0],
    [ 0.3, -1.0,  0.8,  0.1, -0.5],
    [-0.2,  0.7,  0.4, -0.8,  0.3],
    [ 0.6, -0.3, -0.1,  0.9,  0.2],
    [ 0.1,  0.4, -0.6,  0.3, -0.9],
])

print(f"입력 x  : {x}")
print(f"γ (gamma): {gamma}")
print(f"β (beta) : {beta}")
print(f"W shape  : {W.shape}")
print()

# ─── 방법 1: 원래 방식 ───────────────────────────────────
print("📌 방법 1: 원래 방식")
print("   계산: x → LN(γ,β) → W 곱")
print("-" * 45)

x_hat  = layer_norm(x)            # ① γ,β 없이 정규화
x_ln   = gamma * x_hat + beta     # ② γ 곱, β 더하기
result1 = W @ x_ln                # ③ W 행렬 곱

print(f"  x_hat  (정규화만): {np.round(x_hat, 4)}")
print(f"  γ×x̂+β (γ,β 적용): {np.round(x_ln, 4)}")
print(f"  W @ (γ×x̂+β):     {np.round(result1, 4)}")
print()

# ─── 방법 2: 가중치 접기 ──────────────────────────────────
print("📌 방법 2: 가중치 접기")
print("   핵심 수식: W@(γ×x̂+β) = (W×diag(γ))@x̂ + W@β")
print("-" * 45)

# ─ 배포 전 딱 한 번만 계산하는 것들 ─
# np.diag(gamma): gamma 원소를 대각에 갖는 행렬
# W @ np.diag(gamma): W의 j번째 열에 gamma[j]를 곱하는 것과 동일
W_folded = W @ np.diag(gamma)   # γ를 W에 "접어넣기"
b_folded = W @ beta             # β 효과를 bias로 미리 계산

print(f"  [배포 전 한 번만]")
print(f"  W_folded = W @ diag(γ)  shape={W_folded.shape}")
print(f"  b_folded = W @ β      = {np.round(b_folded, 4)}")
print()

# ─ 매 추론마다 ─
x_hat_only = layer_norm(x)                           # γ,β 없이 정규화만
result2    = W_folded @ x_hat_only + b_folded         # W_folded 사용

print(f"  [매 추론마다]")
print(f"  x̂ (정규화만):               {np.round(x_hat_only, 4)}")
print(f"  W_folded @ x̂ + b_folded:  {np.round(result2, 4)}")
print()

# ─── 검증 ─────────────────────────────────────────────────
print("📌 검증: 두 결과가 수학적으로 동일한가?")
print("-" * 45)
print(f"  방법 1: {np.round(result1, 6)}")
print(f"  방법 2: {np.round(result2, 6)}")
print(f"  차이:   {np.round(np.abs(result1 - result2), 10)}")
print(f"  동일?   {np.allclose(result1, result2)} ✅")
print()

# ─── 왜 빠른가 요약 ────────────────────────────────────────
print("📌 왜 빨라지는가? (연산 횟수 비교)")
print("-" * 45)
print(f"  원래 방식:     ① 정규화 → ② γ 원소별 곱({d}번) → ③ β 덧셈({d}번) → ④ W 행렬곱")
print(f"  가중치 접기:   ① 정규화 → ② W_folded 행렬곱  (② ③ 두 단계 제거!)")
print()
print("  → d=4096 (실제 LLM 차원)이면 매 추론마다 4096번의 곱셈·덧셈을 절약!")
print("  → 배치 크기가 크고 시퀀스가 길수록 효과가 더 커집니다.")
print()
print("⚠️  주의: 가중치 접기는 학습 완료 후 배포 단계에서만 적용!")
print("         학습 중엔 γ가 계속 업데이트되므로 사용 불가.")


---
## 📋 핵심 내용 요약

### 1. 왜 정규화가 필요한가?

깊은 네트워크에서 층을 거칠수록 값의 분포가 불안정해집니다.
레이어 정규화는 매 층에서 평균≈0, 분산≈1을 유지시켜 학습을 안정화합니다.

---

### 2. 레이어 정규화 공식

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

| 기호 | 역할 | 초기값 |
|------|------|--------|
| μ | 특징 차원 평균 | — |
| σ² | 특징 차원 분산 | — |
| ε | 0 나누기 방지 (1e-5) | — |
| γ | 학습 가능 스케일 | **1** |
| β | 학습 가능 이동 | **0** |

---

### 3. RMSNorm

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$

- 평균 빼기 생략 → 더 빠름
- LLaMA, Gemma 등 최신 LLM에서 사용

---

### 4. Pre-LN vs Post-LN

| | Post-LN | **Pre-LN** |
|--|---------|--------|
| LN 위치 | 잔차 연결 **후** | 서브레이어 **전** |
| 그래디언트 | LN을 통과 | 잔차로 **직접 흐름** |
| 안정성 | 낮음 | **높음** ✅ |
| 대표 모델 | 원조 Transformer | GPT-2, LLaMA 등 |

---

### 5. 가중치 접기 (추론 최적화)

```python
# 배포 전 한 번만:
W_folded = W @ np.diag(gamma)   # γ를 W에 흡수
b_folded = W @ beta             # β를 bias로 미리 계산

# 매 추론마다:
output = W_folded @ layer_norm(x) + b_folded  # γ, β 별도 연산 없음!
```

---

### 🔗 다음 단계

1. **실제 PyTorch 구현 확인**: `torch.nn.LayerNorm` vs 우리의 numpy 구현 비교
2. **Attention과 결합**: 트랜스포머 블록 전체 구현 (`adv-1-2`)
3. **실제 모델 코드 읽기**: HuggingFace LLaMA에서 RMSNorm 찾아보기

---
*Tutorial: 레이어 정규화: 훈련 안정화 | Section: 레이어 정규화의 수학과 효과*
